# 🚀 GNOT RF Cavity — Kaggle 2×T4 GPU Eğitim Pipeline

Bu notebook, GNOT modelini Kaggle'ın **2×T4 GPU** ortamında **DDP (Data Distributed Parallel)** ile eğitmek için hazırlanmıştır.

## ⚡ Önemli Notlar
- **Accelerator**: GPU T4 ×2 seçili olmalıdır (Settings → Accelerator)
- **DDP Stratejisi**: Her GPU modelin bir kopyasını alır, verinin farklı dilimlerini işler
- **Effective Batch Size**: `batch_size × num_GPUs = 8 × 2 = 16`
- **Eğitimi `!python train.py` ile çalıştırın** (notebook cell'inde değil!)

## 1. 🔧 Kurulum ve Repo Klonlama

In [ ]:
from getpass import getpass
import os

# GitHub Token
token = getpass('GitHub Personal Access Token (PAT) giriniz: ')

# Repo klonla
repo_url = f"https://{token}@github.com/KorayGokceler/rf_cavity_neural_operator.git"
!git clone {repo_url}
%cd rf_cavity_neural_operator

# Sistem bağımlılıkları (gmsh için)
!apt-get install -y libglu1-mesa libxcursor1 libxinerama1 libxft2 libxrender1 --quiet

# Python kütüphaneleri
!pip install -r requirements.txt

In [ ]:
# Son güncellemeleri çek
!git pull origin main
print("✅ Güncellemeler alındı!")

## 2. 🖥️ GPU Doğrulama

Kaggle'da 2 GPU'nun görünüp görünmediğini doğrulayalım.

In [ ]:
import torch

n_gpus = torch.cuda.device_count()
print(f"\n{'='*50}")
print(f"  GPU Sayısı: {n_gpus}")
print(f"{'='*50}")

for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}")
    print(f"    VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"    Compute Capability: {props.major}.{props.minor}")

if n_gpus >= 2:
    print(f"\n✅ {n_gpus} GPU algılandı — DDP hazır!")
elif n_gpus == 1:
    print(f"\n⚠️ Sadece 1 GPU — DDP yerine tek GPU ile devam edilecek.")
    print(f"   Kaggle Settings → Accelerator → GPU T4 ×2 seçtiğinizden emin olun!")
else:
    print(f"\n❌ GPU bulunamadı! Settings → Accelerator kontrol edin.")

## 3. 📦 Veri Seti Yükleme

Kaggle Dataset olarak yüklediyseniz veya sıfırdan üretmek istiyorsanız:

In [ ]:
import os

# === SEÇENEK A: Kaggle Dataset'ten kopyala ===
# Kendi dataset'inizi Kaggle Dataset olarak yüklediyseniz:
# (Kaggle dataset adınıza göre güncelleyin)
#
# kaggle_dataset_path = "/kaggle/input/rf-cavity-dataset/gnot_dataset.h5"
# if os.path.exists(kaggle_dataset_path):
#     os.makedirs("data", exist_ok=True)
#     !cp {kaggle_dataset_path} data/gnot_dataset.h5
#     print("✅ Dataset kopyalandı!")
# else:
#     print(f"❌ Dataset bulunamadı: {kaggle_dataset_path}")

# === SEÇENEK B: Sıfırdan üret (data/ içinde yoksa) ===
if not os.path.exists("data/gnot_dataset.h5"):
    print("Dataset bulunamadı, sıfırdan üretiliyor...")
    # 1) H5 dataset üret (1000 sample, random mode)
    !python src/data_gen/dataset_generator.py --h5_filename rf_cavity_1000_dataset.h5 --n_total 1000 --n_plot 100 --mode random
    # 2) H5'i GNOT PKL formatına dönüştür
    !python convert.py --h5_filepath rf_cavity_1000_dataset.h5 --output_pkl data/gnot_dataset.h5 --modes 0 1 2
else:
    print("✅ Dataset mevcut!")

# Boyut kontrolü
if os.path.exists("data/gnot_dataset.h5"):
    size_mb = os.path.getsize("data/gnot_dataset.h5") / 1e6
    print(f"Dataset boyutu: {size_mb:.1f} MB")

## 4. 🏋️ Eğitim — 2×T4 GPU DDP

### ⚠️ ÖNEMLİ

DDP eğitimi mutlaka `!python train.py` komutuyla çalıştırılmalıdır. Notebook cell'inde `trainer.fit()` çağrısı DDP fork sorunlarına yol açabilir.

### Batch Size Stratejisi
| Config | Batch/GPU | Effective Batch | LR |
|--------|-----------|----------------|----||
| `kaggle_2gpu.yaml` (varsayılan) | 8 | 16 | 1e-3 |
| Büyük batch istiyorsan | 16 | 32 | ~1.4e-3 |

In [ ]:
# 🚀 2-GPU DDP Eğitim Başlat
!python train.py --config configs/kaggle_2gpu.yaml

In [ ]:
# === ALTERNATIF: Daha büyük batch + lr scaling ile ===
# !python train.py --config configs/kaggle_2gpu.yaml \
#     --override training.batch_size=16 training.learning_rate=1.4e-3

In [ ]:
# === HIZLI TEST: Pipeline'ın çalıştığını doğrula ===
# !python train.py --config configs/kaggle_2gpu.yaml --fast_dev_run

## 5. 📊 TensorBoard İzleme

In [ ]:
%load_ext tensorboard
%tensorboard --logdir training_logs

## 6. 📥 Checkpoint Kaydetme

Eğitim tamamlandıktan sonra en iyi checkpoint'ı indirin:

In [ ]:
import glob

# En iyi checkpoint'ı bul
ckpts = glob.glob("training_logs/gnot_kaggle_2gpu/**/best-*.ckpt", recursive=True)
if ckpts:
    best_ckpt = sorted(ckpts, key=os.path.getmtime)[-1]
    print(f"✅ En iyi checkpoint: {best_ckpt}")
    
    # Kaggle output'a kopyala
    !cp {best_ckpt} /kaggle/working/best_model.ckpt
    print(f"Checkpoint kopyalandı: /kaggle/working/best_model.ckpt")
else:
    print("❌ Checkpoint bulunamadı!")

## 7. 🔍 Inference Test

In [ ]:
# En iyi checkpoint ile inference çalıştır
if ckpts:
    !python infer.py --config configs/kaggle_2gpu.yaml --checkpoint {best_ckpt}
else:
    print("Önce eğitimi çalıştırın!")